# Method 1: Random Forest Regressor

**Owner:** Varun Mange

Predict each driver's finishing position from pre-race features, then rank the predictions within each race to get a valid finishing order.

This notebook sets up the data, splits, feature encoding, and hyperparameter grid. The data setup is identical in both method notebooks so the comparison is fair. Sections marked **TODO** are left for model training and analysis.

## 1. Setup

### 1.1 Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import RandomForestRegressor

import f1_utils as f1

### 1.2 Plot theme

In [2]:
f1.set_f1_theme()
NOTEBOOK = "method-01-varun"  # figures are saved to figures/method-01-varun/

### 1.3 Seeds

Final results are reported as mean and std over these five seeds (splits stay the same).

In [3]:
SEEDS = f1.SEEDS
SEEDS

[1, 7, 42, 99, 2026]

## 2. Data

### 2.1 Load CSV

In [4]:
df = f1.load_data()
print(df.shape)
df.head()

(5105, 53)


,row_id,raceId,year,round,date,race_name,driverId,driver_ref,constructorId,constructor_ref,...,circuit_safety_car_rate,circuit_safety_car_avg,circuit_red_flag_rate,race_time_of_day,air_temp,wind_speed,humidity,rain,finish_position,dnf
0,900_1,900,2014,1,2014-03-16,Australian Grand Prix,1,hamilton,131,mercedes,...,0.555556,1.055556,0.055556,day,19.6,18.7,42.0,0.0,19,1
1,900_817,900,2014,1,2014-03-16,Australian Grand Prix,817,ricciardo,9,red_bull,...,0.555556,1.055556,0.055556,day,19.6,18.7,42.0,0.0,22,1
2,900_3,900,2014,1,2014-03-16,Australian Grand Prix,3,rosberg,131,mercedes,...,0.555556,1.055556,0.055556,day,19.6,18.7,42.0,0.0,1,0
3,900_825,900,2014,1,2014-03-16,Australian Grand Prix,825,kevin_magnussen,1,mclaren,...,0.555556,1.055556,0.055556,day,19.6,18.7,42.0,0.0,2,0
4,900_4,900,2014,1,2014-03-16,Australian Grand Prix,4,alonso,6,ferrari,...,0.555556,1.055556,0.055556,day,19.6,18.7,42.0,0.0,4,0


### 2.2 Column groups

- **ID columns**: kept for ranking and analysis, **not** used as features
- **Label columns**: `finish_position` is the target; `dnf` is only for analysis
- **Categorical features**: one-hot encoded
- **Numeric features**: everything else (grid, qualifying, standings, history, circuit, weather)

In [5]:
ID_COLS = [
    "row_id", "raceId", "year", "date", "race_name", "driverId", "constructorId", "circuitId",
]
LABEL_COLS = ["finish_position", "dnf"]
CAT_COLS = ["circuit_type", "race_time_of_day", "driver_ref", "constructor_ref", "circuit_ref"]
NUM_COLS = [c for c in df.columns if c not in ID_COLS + LABEL_COLS + CAT_COLS]

FEATURE_COLS = NUM_COLS + CAT_COLS
TARGET = f1.TARGET

print(f"{len(NUM_COLS)} numeric features:", NUM_COLS)
print(f"{len(CAT_COLS)} categorical features:", CAT_COLS)

38 numeric features: ['round', 'grid', 'quali_position', 'last_quali_session', 'quali_gap_to_pole_s', 'teammate_quali_gap_s', 'grid_minus_quali', 'driver_champ_pos', 'driver_champ_points', 'driver_champ_wins', 'constructor_champ_pos', 'constructor_champ_points', 'constructor_champ_wins', 'constructor_avg_finish_l3', 'career_races', 'career_wins', 'career_podiums', 'career_avg_finish', 'career_dnf_rate', 'circuit_races', 'circuit_avg_finish', 'circuit_wins', 'circuit_podiums', 'races_with_constructor', 'stint_races_with_constructor', 'avg_finish_l3', 'season_dnf_rate', 'circuit_dnf_rate', 'driver_age_years', 'circuit_alt', 'circuit_position_gain_avg', 'circuit_safety_car_rate', 'circuit_safety_car_avg', 'circuit_red_flag_rate', 'air_temp', 'wind_speed', 'humidity', 'rain']
5 categorical features: ['circuit_type', 'race_time_of_day', 'driver_ref', 'constructor_ref', 'circuit_ref']


### 2.3 Missing values

Missing values are intentional (e.g. no circuit history on a first visit, no form at round 1). See `project-docs/features.md`. They are imputed in step 4, fit on train only.

In [6]:
missing = df[FEATURE_COLS].isna().sum()
missing[missing > 0].sort_values(ascending=False).to_frame("missing rows")

,missing rows
circuit_avg_finish,1045
circuit_dnf_rate,1045
avg_finish_l3,268
season_dnf_rate,268
constructor_avg_finish_l3,244
circuit_position_gain_avg,163
teammate_quali_gap_s,144
last_quali_session,73
quali_gap_to_pole_s,73
career_dnf_rate,41


## 3. Chronological splits

### 3.1 Train / validation / test

| Split | Years | Purpose |
|---|---|---|
| Train | 2014-2023 | Model fitting |
| Validation | 2024 | Hyperparameter tuning |
| Test | 2025 | Final evaluation only (do not tune on it) |

In [7]:
train, val, test = f1.split_by_year(df)

pd.DataFrame({
    name: {"years": f"{d.year.min()}-{d.year.max()}", "races": d.raceId.nunique(), "rows": len(d)}
    for name, d in {"train": train, "val": val, "test": test}.items()
}).T

,years,races,rows
train,2014-2023,204,4147
val,2024-2024,24,479
test,2025-2025,24,479


### 3.2 Features and target

In [8]:
X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_val, y_val = val[FEATURE_COLS], val[TARGET]
X_test, y_test = test[FEATURE_COLS], test[TARGET]

print("train:", X_train.shape, " val:", X_val.shape, " test:", X_test.shape)

train: (4147, 43)  val: (479, 43)  test: (479, 43)


## 4. Feature encoding

### 4.1 Encoding pipeline

- **Numeric**: median imputation, plus a 0/1 indicator column for every feature that had missing values in train (so the model can tell "no history" apart from a real value)
- **Categorical**: one-hot encoding (unseen categories in val/test are ignored)
- Fit on **train only**, then transform val and test to avoid leakage

In [9]:
preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median", add_indicator=True), NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
])
preprocess.set_output(transform="pandas")

X_train_enc = preprocess.fit_transform(X_train)
X_val_enc = preprocess.transform(X_val)
X_test_enc = preprocess.transform(X_test)
feature_names = preprocess.get_feature_names_out()

### 4.2 Check the encoded matrices

In [10]:
for name, X in {"train": X_train_enc, "val": X_val_enc, "test": X_test_enc}.items():
    print(f"{name:5s} shape={X.shape}  NaNs={int(X.isna().sum().sum())}")

X_train_enc.head()

train shape=(4147, 163)  NaNs=0
val   shape=(479, 163)  NaNs=0
test  shape=(479, 163)  NaNs=0


,num__round,num__grid,num__quali_position,num__last_quali_session,num__quali_gap_to_pole_s,num__teammate_quali_gap_s,num__grid_minus_quali,num__driver_champ_pos,num__driver_champ_points,num__driver_champ_wins,...,cat__circuit_ref_sepang,cat__circuit_ref_shanghai,cat__circuit_ref_silverstone,cat__circuit_ref_sochi,cat__circuit_ref_spa,cat__circuit_ref_suzuka,cat__circuit_ref_vegas,cat__circuit_ref_villeneuve,cat__circuit_ref_yas_marina,cat__circuit_ref_zandvoort
0,1.0,1.0,1.0,3.0,0.000,-0.364,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,2.0,2.0,3.0,0.317,-0.140,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,3.0,3.0,3.0,0.364,0.364,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,4.0,4.0,3.0,1.514,1.308,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,5.0,5.0,3.0,1.588,1.325,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5. Hyperparameters

### 5.1 Candidate grid

Tuned on the **2024 validation set**: number of trees, maximum tree depth, minimum samples per leaf, and features per split. Edit the values below as needed.

In [11]:
PARAM_GRID = {
    "n_estimators": [200, 500],          # number of trees
    "max_depth": [None, 10, 20],         # None = grow until leaves are pure
    "min_samples_leaf": [1, 5, 10],      # larger = smoother, less overfitting
    "max_features": ["sqrt", 0.5],       # features considered at each split
}

print(len(ParameterGrid(PARAM_GRID)), "combinations")

36 combinations


### 5.2 Starting values

Reasonable defaults to get a first result before running the full search.

In [12]:
DEFAULT_PARAMS = {
    "n_estimators": 500,
    "max_depth": None,
    "min_samples_leaf": 5,
    "max_features": 0.5,
}

## 6. Training (TODO)

### 6.1 Helper: predictions to race ranks

The model predicts a continuous position. `rank_predictions` turns it into a valid finishing order 1..N within each race (same as the baseline).

In [13]:
def rank_predictions(split_df, raw_pred):
    out = split_df.copy()
    out["pred_score"] = raw_pred
    out["pred_position"] = f1.to_race_ranks(out, "pred_score")
    return out

### 6.2 Tune on the validation set (2024)

In [14]:
# TODO: loop over the grid, fit on train, score on val, keep the best params
#
# tuning_results = []
# for params in ParameterGrid(PARAM_GRID):
#     model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
#     model.fit(X_train_enc, y_train)
#     val_pred = rank_predictions(val, model.predict(X_val_enc))
#     tuning_results.append({**params, **f1.evaluate(val_pred, "pred_position")})
#
# tuning_df = pd.DataFrame(tuning_results).sort_values("mae")
# tuning_df.head()

### 6.3 Final runs over seeds

In [15]:
BEST_PARAMS = DEFAULT_PARAMS  # TODO: replace with the best params from 6.2

seed_results = {"val": [], "test": []}
for seed in SEEDS:
    model = ...  # TODO: RandomForestRegressor(**BEST_PARAMS, random_state=seed, n_jobs=-1)
    # model.fit(X_train_enc, y_train)
    # val_pred = rank_predictions(val, model.predict(X_val_enc))
    # test_pred = rank_predictions(test, model.predict(X_test_enc))
    # seed_results["val"].append(f1.evaluate(val_pred, "pred_position"))
    # seed_results["test"].append(f1.evaluate(test_pred, "pred_position"))

## 7. Evaluation and analysis (TODO)

### 7.1 Validation results (tuning)

In [16]:
# TODO: plot how MAE / Spearman change with the main hyperparameters

### 7.2 Test results (mean and std over seeds)

In [17]:
# TODO
# summary = pd.concat({name: f1.summarize_seeds(r) for name, r in seed_results.items()}, axis=1)
# summary.round(3)

### 7.3 Comparison with the baseline

Baseline metrics come from `baseline.ipynb` (saved to `results/baseline_metrics.csv`).

In [18]:
baseline = pd.read_csv(f1.PROJECT_ROOT / "results" / "baseline_metrics.csv", index_col=0)
baseline.round(3)

,accuracy,mae,podium_recall,spearman
train,0.153,3.677,0.693,0.604
val,0.154,2.896,0.667,0.730
test,0.184,3.344,0.750,0.651


### 7.4 Feature importance

In [19]:
# TODO
# e.g. pd.Series(model.feature_importances_, index=feature_names).sort_values().tail(15)

### 7.5 Error analysis

In [20]:
# TODO: e.g. MAE by starting position, finishers vs DNFs, worst races in 2025